# Imports

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import utils.UVA as UVA
import utils.MVA as MVA
import utils.PCA as PCA

from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.decomposition import PCA as skPCA
from sklearn.compose import ColumnTransformer

from sklearn.cluster import KMeans, DBSCAN, MiniBatchKMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples

from sklearn.metrics.cluster import adjusted_rand_score

In [77]:
# import dataframes. Names = df + max date of orders
df_2018_10 = pd.read_csv('data/customers_2016-09-04_2018-10-17.csv')
df_2018_08 = pd.read_csv('data/customers_2016-09-04_2018-08-17.csv')
df_2018_06 = pd.read_csv('data/customers_2016-09-04_2018-06-17.csv')
df_2018_04 = pd.read_csv('data/customers_2016-09-04_2018-04-17.csv')
df_2018_02 = pd.read_csv('data/customers_2016-09-04_2018-02-17.csv')
df_2017_12 = pd.read_csv('data/customers_2016-09-04_2017-12-17.csv')
df_2017_10 = pd.read_csv('data/customers_2016-09-04_2017-10-17.csv')

In [78]:
print(df_2018_10.shape)
print(df_2018_08.shape)
print(df_2018_06.shape)
print(df_2018_04.shape)
print(df_2018_02.shape)
print(df_2017_12.shape)
print(df_2017_10.shape)

(94700, 15)
(93127, 15)
(79835, 15)
(67011, 15)
(53663, 15)
(41324, 15)
(28621, 15)


# Select the same unique points / customer_unique_id

In [ ]:
# Use customer_unique_id from the older dataset as the basis / filter for the future datasets

In [95]:
def transform_df(transform_df, base_df, col):
    transform_df = transform_df[transform_df[col].isin(base_df[col])].sort_values(by=col).set_index(col)
    #transform_df = transform_df[transform_df[col].isin(base_df[col])].sort_values(by=col).set_index(col).reset_index(drop=True)
    
    return transform_df

In [98]:
df_2018_10 = transform_df(df_2018_10, df_2017_10, 'customer_unique_id')
df_2018_08 = transform_df(df_2018_08, df_2017_10, 'customer_unique_id')
df_2018_06 = transform_df(df_2018_06, df_2017_10, 'customer_unique_id')
df_2018_04 = transform_df(df_2018_04, df_2017_10, 'customer_unique_id')
df_2018_02 = transform_df(df_2018_02, df_2017_10, 'customer_unique_id')
df_2017_12 = transform_df(df_2017_12, df_2017_10, 'customer_unique_id')
df_2017_10 = transform_df(df_2017_10, df_2017_10, 'customer_unique_id')

In [99]:
print(df_2018_10.shape)
print(df_2018_08.shape)
print(df_2018_06.shape)
print(df_2018_04.shape)
print(df_2018_02.shape)
print(df_2017_12.shape)
print(df_2017_10.shape)

(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)


In [100]:
df_2017_10.head()

,recency,baby,computers,fixed_telephony,furniture_decor,musical_instruments,sports_leisure,telephony,toys,other_categories,freight_value,frequency_medium,mean_review_score,ratio_comment_message
customer_unique_id,,,,,,,,,,,,,,
0000f46a3911fa3c0805444483337064,221,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,69.0,17.22,0,4,0.0
0000f6ccb0745a6a4b88665a16c9f078,5,0.0,0.0,0.0,0.0,0.0,0.0,25.99,0.0,0.0,17.63,0,5,100.0
0005e1862207bf6ccc02e4228effd9a0,227,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,135.0,15.12,0,4,0.0
0006fdc98a402fceb4eb0ee528f6a8d4,91,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,15.10,0,1,100.0
000a5ad9c4601d2bbdd9ed765d5213b3,67,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,14.29,1,4,0.0


# train kmeans_8 all datasets


In [103]:
def customTransform(df):
    # sélection colonnes avec skewness élevée :
    skewness = df.select_dtypes(include=np.number).skew()
    treshold_skew = 1.0
    cols_skewed = skewness[skewness > treshold_skew].index.tolist()
    cols_unskewed = skewness[skewness < treshold_skew].index.tolist()

    print("Colonnes numériques avec une skewness supérieure à la limite :")
    print(cols_skewed)
    print(cols_unskewed)
    
    # Création du ColumnTransformer
    transformer = ColumnTransformer([
        #('minmax', MinMaxScaler(), cols_skewed),
        ('power', PowerTransformer(), cols_skewed),
        ('standard', StandardScaler(), cols_unskewed)
    ])
    
    X_std = pd.DataFrame(transformer.fit_transform(df), columns=df.columns)
    return X_std

In [105]:
X_2018_10 = customTransform(df_2018_10)
X_2018_08 = customTransform(df_2018_08)
X_2018_06 = customTransform(df_2018_06)
X_2018_04 = customTransform(df_2018_04)
X_2018_02 = customTransform(df_2018_02)
X_2017_12 = customTransform(df_2017_12)
X_2017_10 = customTransform(df_2017_10)

Colonnes numériques avec une skewness supérieure à la limite :
['baby', 'computers', 'fixed_telephony', 'furniture_decor', 'musical_instruments', 'sports_leisure', 'telephony', 'toys', 'other_categories', 'freight_value', 'frequency_medium']
['recency', 'mean_review_score', 'ratio_comment_message']
Colonnes numériques avec une skewness supérieure à la limite :
['baby', 'computers', 'fixed_telephony', 'furniture_decor', 'musical_instruments', 'sports_leisure', 'telephony', 'toys', 'other_categories', 'freight_value', 'frequency_medium']
['recency', 'mean_review_score', 'ratio_comment_message']
Colonnes numériques avec une skewness supérieure à la limite :
['baby', 'computers', 'fixed_telephony', 'furniture_decor', 'musical_instruments', 'sports_leisure', 'telephony', 'toys', 'other_categories', 'freight_value', 'frequency_medium']
['recency', 'mean_review_score', 'ratio_comment_message']
Colonnes numériques avec une skewness supérieure à la limite :
['baby', 'computers', 'fixed_telephon

In [106]:
print(X_2018_10.shape)
print(X_2018_08.shape)
print(X_2018_06.shape)
print(X_2018_04.shape)
print(X_2018_02.shape)
print(X_2017_12.shape)
print(X_2017_10.shape)

(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)
(28621, 14)


In [107]:
X_2018_10.head()

,recency,baby,computers,fixed_telephony,furniture_decor,musical_instruments,sports_leisure,telephony,toys,other_categories,freight_value,frequency_medium,mean_review_score,ratio_comment_message
0,-0.172817,-0.054577,-0.064068,-0.289837,-0.07403,-0.30453,-0.220932,-0.234928,1.847746,-0.140194,-0.684935,1.234801,0.704966,-0.863793
1,-0.172817,-0.054577,-0.064068,-0.289837,-0.07403,-0.30453,4.526283,-0.234928,-0.543709,-0.092037,-0.684935,-1.214155,0.704966,1.170283
2,-0.172817,-0.054577,-0.064068,-0.289837,-0.07403,-0.30453,-0.220932,-0.234928,1.871149,-0.413774,-0.684935,1.302828,0.704966,-0.863793
3,-0.172817,-0.054577,-0.064068,-0.289837,-0.07403,-0.30453,-0.220932,-0.234928,-0.543709,-0.416624,-0.684935,-0.239108,0.704966,1.170283
4,-0.172817,-0.054577,-0.064068,-0.289837,-0.07403,-0.30453,-0.220932,-0.234928,-0.543709,-0.536485,1.455248,-0.511214,-0.014453,-0.863793


In [108]:
kmeans_8_X_2018_10 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2018_10)
kmeans_8_X_2018_08 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2018_08)
kmeans_8_X_2018_06 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2018_06)
kmeans_8_X_2018_04 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2018_04)
kmeans_8_X_2018_02 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2018_02)
kmeans_8_X_2017_12 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2017_12)
kmeans_8_X_2017_10 = KMeans(n_clusters=8, verbose=1, random_state=0).fit(X_2017_10)

Initialization complete
Iteration 0, inertia 274612.9238011333.
Iteration 1, inertia 210117.52235608926.
Iteration 2, inertia 206249.96794172042.
Iteration 3, inertia 204042.3880502939.
Iteration 4, inertia 204019.01334120307.
Iteration 5, inertia 204017.38890518525.
Converged at iteration 5: center shift 1.9497034298066814e-05 within tolerance 9.999999999999561e-05.
Initialization complete
Iteration 0, inertia 278618.57782481.
Iteration 1, inertia 202708.5286695574.
Iteration 2, inertia 202694.0297382821.
Converged at iteration 2: strict convergence.
Initialization complete
Iteration 0, inertia 306873.9073390304.
Iteration 1, inertia 227429.53391114573.
Iteration 2, inertia 225766.6990362496.
Iteration 3, inertia 223650.66254711576.
Iteration 4, inertia 222309.22199300898.
Iteration 5, inertia 221580.01934126025.
Iteration 6, inertia 220877.1274107091.
Iteration 7, inertia 220029.85281800688.
Iteration 8, inertia 219376.3989288951.
Iteration 9, inertia 219169.1963208156.
Iteration 10,

In [113]:
# apply labels to first df

X_labels = X_2017_10.copy()
X_labels['km_2018_10'] = kmeans_8_X_2018_10.labels_
X_labels['km_2018_08'] = kmeans_8_X_2018_08.labels_
X_labels['km_2018_06'] = kmeans_8_X_2018_06.labels_
X_labels['km_2018_04'] = kmeans_8_X_2018_04.labels_
X_labels['km_2018_02'] = kmeans_8_X_2018_02.labels_
X_labels['km_2017_12'] = kmeans_8_X_2017_12.labels_
X_labels['km_2017_10'] = kmeans_8_X_2017_10.labels_

In [114]:
X_labels

,recency,baby,computers,fixed_telephony,furniture_decor,musical_instruments,sports_leisure,telephony,toys,other_categories,...,frequency_medium,mean_review_score,ratio_comment_message,km_2018_10,km_2018_08,km_2018_06,km_2018_04,km_2018_02,km_2017_12,km_2017_10
0,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,1.869734,-0.111998,...,-0.669877,-0.090125,-0.860951,3,1,1,1,4,1,2
1,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,4.549617,-0.233766,-0.537311,-0.063104,...,-0.669877,0.672583,1.167715,0,0,0,0,0,0,0
2,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,1.891437,-0.389753,...,-0.669877,-0.090125,-0.860951,3,1,2,1,4,1,2
3,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,-0.537311,-0.392645,...,-0.669877,-2.378249,1.167715,1,1,3,3,6,4,1
4,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,-0.537311,-0.514331,...,1.425484,-0.090125,-0.860951,3,3,2,2,3,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28616,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,1.877899,-0.038656,...,-0.669877,-0.090125,-0.860951,3,1,2,1,4,1,2
28617,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,1.839188,-0.392645,...,-0.669877,-0.090125,1.167715,1,1,3,3,6,2,2
28618,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,-0.537311,3.745472,...,-0.669877,-2.378249,-0.860951,3,1,2,1,4,6,1
28619,-0.17185,-0.05393,-0.063792,-0.286994,-0.073311,-0.302057,-0.219799,-0.233766,-0.537311,0.424710,...,1.425484,-0.852833,-0.860951,3,3,1,2,3,6,6


# Compute ARI